# 10 · Comparar búsqueda híbrida con búsqueda vectorial en Qdrant

Medimos si combinar el vector denso de `nomic-embed-text` con BM25 y fusionar ambos resultados con RRF mejora la recuperación. Se comparan cuatro configuraciones: vectorial e híbrida, cada una con y sin reranking MiniLM-L6. Se mantiene la configuración decidida en las fichas anteriores: troceo por estructura 1500/200 sin ruta, `k = 4` y 20 candidatos cuando hay reranking.

Necesita Qdrant en `localhost:6333`, Ollama con `nomic-embed-text` y la colección creada por `notebooks/07_comparar_troceo_estructura.ipynb`. La colección original solo se lee; la colección híbrida se reconstruye en cada ejecución.

## 1. Dependencias

`fastembed` está declarado en `pyproject.toml`. El resto de dependencias ya se usaba en los notebooks anteriores. Todo se instala con `uv sync`.

## 2. Imports y configuración

Las cuatro configuraciones, modelos, colecciones y umbrales se declaran en esta única celda.

In [1]:
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, Fusion, FusionQuery, Modifier, PointStruct, Prefetch,
    SparseVector, SparseVectorParams, VectorParams,
)
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, evaluar_hibrida, fuentes_ausentes,
    leer_evidencias, leer_golden_set, preparar_query,
    rango_primer_acierto, rango_primer_acierto_evidencia,
    reordenar_por_score, trocear_por_estructura,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CSV = RAIZ / "data" / "resultados" / "hybrid.csv"
NOTEBOOK_INDEXADO = "notebooks/07_comparar_troceo_estructura.ipynb"

MODELO = "nomic"
MAXIMO = 1500
MINIMO = 200
CON_RUTA = False
K = 4
CANDIDATOS = 20
PREFETCH = 20
MODELO_BM25 = "Qdrant/bm25"
RERANKER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOTE_LECTURA = 128
MEJORA_MINIMA = 2
LATENCIA_ANADIDA_MAXIMA_MS = 100
REFERENCIA = "vectorial + reranking"
COMPARADA = "híbrida + reranking"

CONFIGURACIONES = [
    {"nombre": "vectorial", "busqueda": "vectorial", "reranker": None},
    {"nombre": "híbrida", "busqueda": "híbrida", "reranker": None},
    {"nombre": REFERENCIA, "busqueda": "vectorial", "reranker": RERANKER},
    {"nombre": COMPARADA, "busqueda": "híbrida", "reranker": RERANKER},
]


def nombre_coleccion(modelo: str, maximo: int, minimo: int, con_ruta: bool) -> str:
    sufijo = "ruta" if con_ruta else "sinruta"
    return f"pydocs_{modelo}_est{maximo}_m{minimo}_{sufijo}"


COLECCION = nombre_coleccion(MODELO, MAXIMO, MINIMO, CON_RUTA)
COLECCION_HIBRIDA = COLECCION + "_hibrida"
print("Origen:", COLECCION)
print("Híbrida:", COLECCION_HIBRIDA)

W1117 14:32:21.031000 91250 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Origen: pydocs_nomic_est1500_m200_sinruta
Híbrida: pydocs_nomic_est1500_m200_sinruta_hibrida


## 3. Conexión con Qdrant y Ollama

Se comprueban ambos servicios y la presencia de `nomic-embed-text` antes de construir nada.

In [2]:
qdrant = QdrantClient("localhost", port=6333)
try:
    qdrant.get_collections()
except Exception as error:
    raise RuntimeError(
        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d`."
    ) from error

modelo_ollama = MODELOS[MODELO]["nombre"]
try:
    ollama.show(modelo_ollama)
except ConnectionError as error:
    raise RuntimeError("Ollama no responde. Arranca la app o ejecuta `ollama serve`.") from error
except ollama.ResponseError as error:
    raise RuntimeError(
        f"Ollama no tiene {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}`."
    ) from error

print("Qdrant y Ollama disponibles; modelo:", modelo_ollama)

Qdrant y Ollama disponibles; modelo: nomic-embed-text


## 4. Comprobación de la colección de origen

Se verifica que el índice base coincide con el troceo decidido y se guarda su número de puntos para comprobar al final que no cambió.

In [3]:
if not qdrant.collection_exists(COLECCION):
    raise RuntimeError(f"No existe {COLECCION}. Créala con `{NOTEBOOK_INDEXADO}`.")

puntos_origen = qdrant.count(COLECCION, exact=True).count
n_chunks = len(trocear_por_estructura(CARPETA_CORPUS, MAXIMO, MINIMO))
if puntos_origen != n_chunks:
    raise RuntimeError(
        f"{COLECCION} tiene {puntos_origen} puntos y el troceo produce {n_chunks}. "
        f"Recréala con `{NOTEBOOK_INDEXADO}`."
    )

info_origen = qdrant.get_collection(COLECCION)
config_densa = info_origen.config.params.vectors
if isinstance(config_densa, dict):
    if len(config_densa) != 1:
        raise RuntimeError(f"No se puede deducir el vector denso de {COLECCION}: {list(config_densa)}")
    config_densa = next(iter(config_densa.values()))
dimension_densa = config_densa.size
print(f"Origen intacto: {puntos_origen} puntos, dimensión {dimension_densa}")

Origen intacto: 7955 puntos, dimensión 768


## 5. Carga de BM25 y del reranker

Se mide la carga de cada modelo por separado y se hace una llamada de calentamiento fuera de las latencias del experimento.

In [4]:
inicio = time.perf_counter()
try:
    modelo_bm25 = SparseTextEmbedding(MODELO_BM25)
    next(modelo_bm25.embed(["warm up document"]))
    next(modelo_bm25.query_embed("warm up query"))
except Exception as error:
    raise RuntimeError(f"No se pudo cargar el modelo disperso {MODELO_BM25}: {error}") from error
carga_bm25_s = time.perf_counter() - inicio

inicio = time.perf_counter()
try:
    reranker = CrossEncoder(RERANKER)
    reranker.predict([("warm up query", "warm up document")])
except Exception as error:
    raise RuntimeError(f"No se pudo cargar el reranker {RERANKER}: {error}") from error
carga_reranker_s = time.perf_counter() - inicio

print(f"BM25 cargado en {carga_bm25_s:.1f} s")
print(f"Reranker cargado en {carga_reranker_s:.1f} s; dispositivo {reranker.device}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BM25 cargado en 0.1 s
Reranker cargado en 2.0 s; dispositivo mps:0


## 6. Construcción de la colección híbrida

La colección híbrida se recrea con un vector denso con nombre y un vector disperso BM25 con IDF. Los vectores densos, IDs y payloads se copian del índice base; no se vuelven a calcular embeddings con Ollama.

In [5]:
inicio_construccion = time.perf_counter()
if qdrant.collection_exists(COLECCION_HIBRIDA):
    qdrant.delete_collection(COLECCION_HIBRIDA)

qdrant.create_collection(
    collection_name=COLECCION_HIBRIDA,
    vectors_config={"denso": VectorParams(size=dimension_densa, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

offset = None
barra = tqdm(total=puntos_origen, desc="Copiar + BM25")
while True:
    registros, offset_siguiente = qdrant.scroll(
        collection_name=COLECCION,
        offset=offset,
        limit=LOTE_LECTURA,
        with_payload=True,
        with_vectors=True,
    )
    if not registros:
        break

    textos = [registro.payload["texto"] for registro in registros]
    dispersos = list(modelo_bm25.embed(textos, batch_size=LOTE_LECTURA))
    puntos = []
    for registro, disperso in zip(registros, dispersos):
        vector_denso = registro.vector
        if isinstance(vector_denso, dict):
            if len(vector_denso) != 1:
                raise RuntimeError(f"El punto {registro.id} tiene varios vectores densos")
            vector_denso = next(iter(vector_denso.values()))
        puntos.append(
            PointStruct(
                id=registro.id,
                vector={
                    "denso": vector_denso,
                    "bm25": SparseVector(
                        indices=disperso.indices.tolist(),
                        values=disperso.values.tolist(),
                    ),
                },
                payload=registro.payload,
            )
        )
    qdrant.upsert(COLECCION_HIBRIDA, points=puntos, wait=True)
    barra.update(len(puntos))
    if offset_siguiente is None:
        break
    offset = offset_siguiente
barra.close()

construccion_s = time.perf_counter() - inicio_construccion
puntos_hibrida = qdrant.count(COLECCION_HIBRIDA, exact=True).count
if puntos_hibrida != puntos_origen:
    raise RuntimeError(
        f"La colección híbrida tiene {puntos_hibrida} puntos; se esperaban {puntos_origen}."
    )
print(f"Colección híbrida: {puntos_hibrida} puntos en {construccion_s:.1f} s")

Copiar + BM25:   0%|          | 0/7955 [00:00<?, ?it/s]

Colección híbrida: 7955 puntos en 6.9 s


## 7. Preguntas y evidencias

Las preguntas, categorías y evidencias se leen de `docs/`. Si una fuente no existe, se muestra su ID y su ruta.

In [6]:
preguntas = leer_golden_set(RUTA_GOLDEN)
ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)
for id_, ruta in ausentes:
    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")
evidencias = leer_evidencias(RUTA_EVIDENCIAS)
print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())

35 preguntas: {'A': 10, 'B': 10, 'C': 9, 'D': 3, 'E': 3}


## 8. Evaluación de las cuatro configuraciones

La búsqueda vectorial consulta el índice original. La híbrida hace dos prefetch —denso y BM25— y los fusiona con RRF dentro de Qdrant. Con reranking se recuperan 20 candidatos, se puntúan con MiniLM-L6 y se conservan los 4 mejores.

In [7]:
def embeber_denso(texto: str) -> list[float]:
    return ollama.embed(model=modelo_ollama, input=[texto]).embeddings[0]


def embeber_bm25_query(texto: str) -> SparseVector:
    disperso = next(modelo_bm25.query_embed(texto))
    return SparseVector(indices=disperso.indices.tolist(), values=disperso.values.tolist())


resultados = []
for config in CONFIGURACIONES:
    nombre = config["nombre"]
    for pregunta in tqdm(preguntas, desc=nombre):
        limite = CANDIDATOS if config["reranker"] else K
        inicio = time.perf_counter()
        denso = embeber_denso(preparar_query(MODELO, pregunta["query"]))
        if config["busqueda"] == "vectorial":
            respuesta = qdrant.query_points(
                COLECCION, query=denso, limit=limite, with_payload=True
            ).points
            escala = "coseno"
        else:
            disperso = embeber_bm25_query(pregunta["query"])
            respuesta = qdrant.query_points(
                collection_name=COLECCION_HIBRIDA,
                prefetch=[
                    Prefetch(query=denso, using="denso", limit=PREFETCH),
                    Prefetch(query=disperso, using="bm25", limit=PREFETCH),
                ],
                query=FusionQuery(fusion=Fusion.RRF),
                limit=limite,
                with_payload=True,
            ).points
            escala = "RRF"
        busqueda_ms = (time.perf_counter() - inicio) * 1000

        candidatos = [
            {"fuente": punto.payload["fuente"], "texto": punto.payload["texto"]}
            for punto in respuesta
        ]
        rerank_ms = 0.0
        if config["reranker"]:
            inicio = time.perf_counter()
            scores = [
                float(score)
                for score in reranker.predict(
                    [(pregunta["query"], candidato["texto"]) for candidato in candidatos]
                )
            ]
            recuperados = reordenar_por_score(candidatos, scores, K)
            rerank_ms = (time.perf_counter() - inicio) * 1000
            score_primero = max(scores) if scores else None
            escala = "cross-encoder"
        else:
            recuperados = candidatos
            score_primero = float(respuesta[0].score) if respuesta else None

        fila = {
            "configuracion": nombre,
            "busqueda": config["busqueda"],
            "id": pregunta["id"],
            "categoria": pregunta["categoria"],
            "busqueda_ms": busqueda_ms,
            "rerank_ms": rerank_ms,
            "score_primero": score_primero,
            "escala": escala,
            "rango": None,
            "rango_evidencia": None,
        }
        if pregunta["fuentes"]:
            fila["rango"] = rango_primer_acierto(
                [r["fuente"] for r in recuperados], pregunta["fuentes"]
            )
            fila["rango_evidencia"] = rango_primer_acierto_evidencia(
                recuperados, evidencias[pregunta["id"]]
            )
        resultados.append(fila)

df = pd.DataFrame(resultados)
df_abcd = df[df["categoria"] != "E"].copy()
print(len(df), "evaluaciones;", len(df_abcd), "corresponden a A-D")

vectorial:   0%|          | 0/35 [00:00<?, ?it/s]

híbrida:   0%|          | 0/35 [00:00<?, ?it/s]

vectorial + reranking:   0%|          | 0/35 [00:00<?, ?it/s]

híbrida + reranking:   0%|          | 0/35 [00:00<?, ?it/s]

140 evaluaciones; 128 corresponden a A-D


## 9. Métricas por configuración

Se calculan hit@1, hit@4 y MRR por fuente y por evidencia, además de los aciertos contados usados por el criterio.

In [8]:
def metricas(grupo: pd.DataFrame) -> dict:
    salida = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        calculadas = calcular_metricas(rangos, K)
        salida[prefijo + "hit@1"] = calculadas["hit@1"]
        salida[prefijo + f"hit@{K}"] = calculadas[f"hit@{K}"]
        salida[prefijo + "MRR"] = calculadas["mrr"]
    return salida


por_config = pd.DataFrame(
    {
        config["nombre"]: metricas(df_abcd[df_abcd["configuracion"] == config["nombre"]])
        for config in CONFIGURACIONES
    }
).T
n_abcd = df_abcd["id"].nunique()
por_config["aciertos hit@4"] = (por_config["hit@4"] * n_abcd).round().astype(int)
por_config["aciertos ev_hit@4"] = (por_config["ev_hit@4"] * n_abcd).round().astype(int)
por_config.round(3)

,hit@1,hit@4,MRR,ev_hit@1,ev_hit@4,ev_MRR,aciertos hit@4,aciertos ev_hit@4
vectorial,0.719,0.906,0.786,0.406,0.594,0.477,29,19
híbrida,0.688,0.906,0.781,0.375,0.656,0.495,29,21
vectorial + reranking,0.688,0.938,0.797,0.344,0.688,0.495,30,22
híbrida + reranking,0.625,0.938,0.768,0.312,0.719,0.484,30,23


## 10. Desglose por categoría

Se muestran hit@4 y ev_hit@4 para A-D; la categoría B permite comprobar el efecto sobre términos exactos y siglas.

In [9]:
filas_categoria = []
for config in CONFIGURACIONES:
    sub = df_abcd[df_abcd["configuracion"] == config["nombre"]]
    for categoria, grupo in sub.groupby("categoria"):
        m = metricas(grupo)
        filas_categoria.append(
            {
                "configuración": config["nombre"],
                "categoría": categoria,
                "preguntas": len(grupo),
                "hit@4": m["hit@4"],
                "ev_hit@4": m["ev_hit@4"],
                "aciertos ev_hit@4": round(m["ev_hit@4"] * len(grupo)),
            }
        )
por_categoria = pd.DataFrame(filas_categoria)
por_categoria.round(3)

,configuración,categoría,preguntas,hit@4,ev_hit@4,aciertos ev_hit@4
0,vectorial,A,10,0.900,0.500,5
1,vectorial,B,10,0.900,0.700,7
2,vectorial,C,9,1.000,0.778,7
3,vectorial,D,3,0.667,0.000,0
4,híbrida,A,10,0.800,0.500,5
5,híbrida,B,10,1.000,0.800,8
6,híbrida,C,9,1.000,0.889,8
7,híbrida,D,3,0.667,0.000,0
8,vectorial + reranking,A,10,0.900,0.600,6
9,vectorial + reranking,B,10,1.000,0.800,8


## 11. Preguntas sin respuesta y escala del score

Para cada configuración se compara el primer score de las preguntas E con la media y el mínimo de A-D. Coseno, RRF y cross-encoder son escalas distintas y no se comparan entre configuraciones.

In [10]:
filas_e = []
for config in CONFIGURACIONES:
    sub = df[df["configuracion"] == config["nombre"]]
    scores_abcd = sub[sub["categoria"] != "E"]["score_primero"]
    for _, fila in sub[sub["categoria"] == "E"].iterrows():
        filas_e.append(
            {
                "configuración": config["nombre"],
                "escala": fila["escala"],
                "pregunta E": fila["id"],
                "score primero": fila["score_primero"],
                "media A-D": scores_abcd.mean(),
                "mínimo A-D": scores_abcd.min(),
            }
        )
pd.DataFrame(filas_e).round(3)

,configuración,escala,pregunta E,score primero,media A-D,mínimo A-D
0,vectorial,coseno,E01,0.649,0.770,0.685
1,vectorial,coseno,E02,0.667,0.770,0.685
2,vectorial,coseno,E03,0.662,0.770,0.685
3,híbrida,RRF,E01,0.553,0.790,0.500
4,híbrida,RRF,E02,0.500,0.790,0.500
5,híbrida,RRF,E03,1.000,0.790,0.500
6,vectorial + reranking,cross-encoder,E01,-3.540,5.001,-0.389
7,vectorial + reranking,cross-encoder,E02,-6.717,5.001,-0.389
8,vectorial + reranking,cross-encoder,E03,-1.124,5.001,-0.389
9,híbrida + reranking,cross-encoder,E01,-3.540,4.997,-0.389


## 12. Criterio de decisión

La híbrida con reranking se compara con la vectorial con reranking. Se adopta si suma al menos 2 aciertos por evidencia en total o en B, no pierde aciertos por fuente y añade menos de 100 ms de búsqueda.

In [11]:
def aciertos_categoria(configuracion: str, categoria: str, columna: str) -> int:
    sub = df_abcd[
        (df_abcd["configuracion"] == configuracion)
        & (df_abcd["categoria"] == categoria)
    ]
    return sum(1 for rango in sub[columna] if not pd.isna(rango) and int(rango) <= K)


latencia_busqueda = df_abcd.groupby("configuracion")["busqueda_ms"].mean()
ref = por_config.loc[REFERENCIA]
comparada = por_config.loc[COMPARADA]
datos_referencia = {
    "ev_aciertos": int(ref["aciertos ev_hit@4"]),
    "aciertos": int(ref["aciertos hit@4"]),
    "ev_aciertos_b": aciertos_categoria(REFERENCIA, "B", "rango_evidencia"),
    "busqueda_ms": float(latencia_busqueda[REFERENCIA]),
}
datos_hibrida = {
    "ev_aciertos": int(comparada["aciertos ev_hit@4"]),
    "aciertos": int(comparada["aciertos hit@4"]),
    "ev_aciertos_b": aciertos_categoria(COMPARADA, "B", "rango_evidencia"),
    "busqueda_ms": float(latencia_busqueda[COMPARADA]),
}
criterio = evaluar_hibrida(
    datos_referencia, datos_hibrida, MEJORA_MINIMA, LATENCIA_ANADIDA_MAXIMA_MS
)
decision = "adoptar búsqueda híbrida" if criterio["cumple"] else "mantener búsqueda vectorial"
print("Referencia:", datos_referencia)
print("Híbrida:", datos_hibrida)
print("Decisión:", decision)
pd.DataFrame([criterio])

Referencia: {'ev_aciertos': 22, 'aciertos': 30, 'ev_aciertos_b': 8, 'busqueda_ms': 33.516330841848685}
Híbrida: {'ev_aciertos': 23, 'aciertos': 30, 'ev_aciertos_b': 8, 'busqueda_ms': 29.06722400120998}
Decisión: mantener búsqueda vectorial


,mejora_ev,mejora_ev_b,fuente_no_baja,latencia_anadida_ms,latencia_ok,cumple
0,1,0,True,-4.449107,True,False


## 13. Tabla resumen y CSV

Se guarda una fila por configuración en `data/resultados/hybrid.csv`. Las filas vectoriales tienen construcción 0 porque usan el índice existente.

In [12]:
filas_resumen = []
for config in CONFIGURACIONES:
    nombre = config["nombre"]
    sub = df_abcd[df_abcd["configuracion"] == nombre]
    m = por_config.loc[nombre]
    fila = {
        "configuración": nombre,
        "búsqueda": config["busqueda"],
        "modelo de reranking": config["reranker"] or "ninguno",
        "candidatos": CANDIDATOS if config["reranker"] else K,
        "hit@1": m["hit@1"],
        "hit@4": m["hit@4"],
        "MRR": m["MRR"],
        "ev_hit@1": m["ev_hit@1"],
        "ev_hit@4": m["ev_hit@4"],
        "ev_MRR": m["ev_MRR"],
    }
    for categoria in sorted(df_abcd["categoria"].unique()):
        mc = metricas(sub[sub["categoria"] == categoria])
        fila[f"hit@4 {categoria}"] = mc["hit@4"]
        fila[f"ev_hit@4 {categoria}"] = mc["ev_hit@4"]
    fila["latencia búsqueda (ms)"] = sub["busqueda_ms"].mean()
    fila["latencia reranking (ms)"] = sub["rerank_ms"].mean()
    fila["latencia total (ms)"] = fila["latencia búsqueda (ms)"] + fila["latencia reranking (ms)"]
    fila["construcción colección (s)"] = construccion_s if config["busqueda"] == "híbrida" else 0.0
    filas_resumen.append(fila)

resumen = pd.DataFrame(filas_resumen)
assert not resumen.isna().any().any(), "El CSV tendría celdas vacías"
RUTA_CSV.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CSV, index=False)
print(f"Guardado en {RUTA_CSV.relative_to(RAIZ)}")
resumen.round(3)

Guardado en data/resultados/hybrid.csv


,configuración,búsqueda,modelo de reranking,candidatos,hit@1,hit@4,MRR,ev_hit@1,ev_hit@4,ev_MRR,...,hit@4 B,ev_hit@4 B,hit@4 C,ev_hit@4 C,hit@4 D,ev_hit@4 D,latencia búsqueda (ms),latencia reranking (ms),latencia total (ms),construcción colección (s)
0,vectorial,vectorial,ninguno,4,0.719,0.906,0.786,0.406,0.594,0.477,...,0.9,0.7,1.0,0.778,0.667,0.000,49.749,0.000,49.749,0.000
1,híbrida,híbrida,ninguno,4,0.688,0.906,0.781,0.375,0.656,0.495,...,1.0,0.8,1.0,0.889,0.667,0.000,20.183,0.000,20.183,6.914
2,vectorial + reranking,vectorial,cross-encoder/ms-marco-MiniLM-L-6-v2,20,0.688,0.938,0.797,0.344,0.688,0.495,...,1.0,0.8,1.0,0.778,0.667,0.333,33.516,108.914,142.430,0.000
3,híbrida + reranking,híbrida,cross-encoder/ms-marco-MiniLM-L-6-v2,20,0.625,0.938,0.768,0.312,0.719,0.484,...,1.0,0.8,1.0,0.778,0.667,0.333,29.067,111.845,140.913,6.914


## 14. Comprobación final

La colección de origen debe conservar exactamente los mismos puntos; la híbrida debe tener el mismo número.

In [13]:
puntos_origen_final = qdrant.count(COLECCION, exact=True).count
puntos_hibrida_final = qdrant.count(COLECCION_HIBRIDA, exact=True).count
if puntos_origen_final != puntos_origen:
    raise RuntimeError(
        f"La colección original cambió: empezó con {puntos_origen} y terminó con {puntos_origen_final}."
    )
if puntos_hibrida_final != puntos_origen:
    raise RuntimeError(
        f"La colección híbrida terminó con {puntos_hibrida_final}; se esperaban {puntos_origen}."
    )
print(f"Comprobación correcta: origen={puntos_origen_final}, híbrida={puntos_hibrida_final}")

Comprobación correcta: origen=7955, híbrida=7955
